# 01 · `io.py`: lookup tables, YAML mappings and the ontology

`io.py` is the foundation for everything else. It reads three kinds of files
bundled in `nextbrain_utils/lut/`:

| Kind | Files | Loader |
|---|---|---|
| FreeSurfer LUT (`ID NAME R G B A`) | `NextBrainLUT.txt`, `ASegLUT.txt`, `FreeSurferColorLUT.txt`, `SuperSynth*LUT.txt`, `LauraLUT.txt` | `load_lut` → `LUT` |
| Mapping (target name → list of NextBrain IDs) | `NextBrainTo{ASeg,SuperSynth,Laura}.yaml` | `load_yaml` |
| Allen ontology (nested JSON tree) | `AllenBrainOntologyDev.json` (used), `AllenBrainOntology.json` (unused) | `load_ontology` / `load_json` |

`NextBrainLabels.yaml` (a `name: id` dictionary) is also bundled. No code
uses it, but it is useful: it lists the **≈333 labels that NextBrain
segmentations actually contain**. `NextBrainLUT.txt` is a superset with 883
entries, which also covers histology-only labels such as cortical gyri and
Brodmann areas.

In [1]:
import sys, os
from pathlib import Path
sys.path.insert(0, ".")          # nb_helpers.py lives next to the notebooks
WORK = Path("_work/01"); WORK.mkdir(parents=True, exist_ok=True)   # scratch outputs of this notebook

import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib.pyplot as plt
import nb_helpers as H
pd.set_option("display.max_rows", 60, "display.max_colwidth", 70)

In [2]:
from nextbrain_utils import io
LUTDIR = Path(io.__file__).parent / "lut"
print(io.PATH_NEXTBRAIN)
print(io.LUT_DTYPE)

/home/agustin/phd/synthseg_nextbrain/project/dev/wp1/allen_ontology/nextbrain-utils/nextbrain_utils/lut/NextBrainLUT.txt
[('ID', '<i8'), ('NAME', 'S256'), ('R', '<u8'), ('G', '<u8'), ('B', '<u8'), ('A', '<u8')]


## The LUT text format

Each non-comment line has the form `ID  NAME  R  G  B  A`. Anything after `#`
is ignored, and empty lines are skipped. Here are the raw first lines:

In [3]:
print("".join(open(LUTDIR / "NextBrainLUT.txt").readlines()[:6]))
print("".join(open(LUTDIR / "ASegLUT.txt").readlines()[:4]))

0     Unknown                                                              0    0    0    0
1     middle_frontal_gyrus                                                 212  178  53   0
2     frontomarginal_gyrus                                                 238  207  56   0
3     frontal_pole                                                         234  209  128  0
4     medial_subdivision_of_area_10                                        222  189  62   0
5     lateral_subdivision_of_area_10                                       222  189  62   0

#No.  Label Name:                             R   G   B   A
0     Unknown                                 0   0   0   0
2     Left-Cerebral-White-Matter              245 245 245 0
3     Left-Cerebral-Cortex                    205 62  78  0



## `load_lut` and the `LUT` class

`load_lut` parses the file into a NumPy **structured array** with fields
`ID, NAME, R, G, B, A`, then `.view(LUT)` turns it into a `LUT` (a subclass of
`np.ndarray`). So you can treat it as an array (filter, mask, sort) and also
call convenience methods on it.

Note that `NAME` is stored as **bytes** (`S256`). The methods decode it for you,
but you must encode names yourself when you index the raw field
(`lut["NAME"] == b"putamen"`).

In [4]:
lut = io.load_lut()          # default = NextBrainLUT.txt
print(type(lut), lut.shape, lut.dtype.names)
lut[:5]

<class 'nextbrain_utils.io.LUT'> (883,) ('ID', 'NAME', 'R', 'G', 'B', 'A')


LUT([(0, b'Unknown',   0,   0,   0, 0),
     (1, b'middle_frontal_gyrus', 212, 178,  53, 0),
     (2, b'frontomarginal_gyrus', 238, 207,  56, 0),
     (3, b'frontal_pole', 234, 209, 128, 0),
     (4, b'medial_subdivision_of_area_10', 222, 189,  62, 0)],
    dtype=[('ID', '<i8'), ('NAME', 'S256'), ('R', '<u8'), ('G', '<u8'), ('B', '<u8'), ('A', '<u8')])

In [5]:
print("labels[:8] :", lut.labels[:8])
print("names[:4]  :", lut.names[:4])
print()
print("label2name(79)        ->", lut.label2name(79))
print("name2label('putamen') ->", lut.name2label("putamen"))
print()
# Without an argument, each method returns the whole dictionary
d = lut.label2name()
print(type(d), len(d), list(d.items())[:3])

labels[:8] : [0, 1, 2, 3, 4, 5, 6, 7]
names[:4]  : ['Unknown', 'middle_frontal_gyrus', 'frontomarginal_gyrus', 'frontal_pole']

label2name(79)        -> putamen
name2label('putamen') -> 79

<class 'dict'> 883 [(0, 'Unknown'), (1, 'middle_frontal_gyrus'), (2, 'frontomarginal_gyrus')]


### ⚠️ Issue 10: the colour accessors fail with NumPy 2

`colors`, `label2color` and `name2color` all run
`self[["R","G","B","A"]].astype("u1")`. That casts a *multi-field structured*
array to a plain `uint8` array, which NumPy refuses (`TypeError`). No
converter uses these methods, so only direct users are affected. Workaround:

In [6]:
print("numpy", np.__version__)
for fn in ["colors", "label2color(79)", "name2color('putamen')"]:
    try:
        eval(f"lut.{fn}")
    except TypeError:
        print(f"lut.{fn:22s} -> TypeError")

from numpy.lib.recfunctions import structured_to_unstructured
colors = structured_to_unstructured(lut[["R", "G", "B", "A"]]).astype("u1")
print("workaround colors[:3]:\n", colors[:3])
print("putamen RGBA:", tuple(colors[lut["ID"] == 79][0]))

numpy 2.4.6
lut.colors                 -> TypeError
lut.label2color(79)        -> TypeError
lut.name2color('putamen')  -> TypeError
workaround colors[:3]:
 [[  0   0   0   0]
 [212 178  53   0]
 [238 207  56   0]]
putamen RGBA: (np.uint8(168), np.uint8(180), np.uint8(133), np.uint8(0))


Looking up a label that is not in the LUT raises an `IndexError`, because the
method indexes `[0]` into an empty selection. There is no friendly error:

In [7]:
try:
    lut.label2name(999)
except IndexError as e:
    print("IndexError:", e)

IndexError: index 0 is out of bounds for axis 0 with size 0


## How NextBrain IDs are organised

* `0`: Unknown
* `1 … 846`: **subcortical, white-matter, cerebellar and brainstem** structures.
  The names are Allen ontology names in snake_case.
* `2000 … 2035`: **DK cortical** parcels, using the *right-hemisphere* IDs.
  NextBrain gives both hemispheres the 2xxx IDs; only the separate sides mask
  says which hemisphere a voxel belongs to.

In [8]:
df = pd.DataFrame({"ID": lut.labels, "NAME": lut.names})
print(len(df), "labels;  ID range", df.ID.min(), "-", df.ID.max())
print("subcortical (<1000):", (df.ID.between(1, 999)).sum(), "   cortical (>=2000):", (df.ID >= 2000).sum())
df[df.ID >= 2000].head(8)

883 labels;  ID range 0 - 2035
subcortical (<1000): 846    cortical (>=2000): 36


,ID,NAME
847,2000,ctx-unknown
848,2001,ctx-bankssts
849,2002,ctx-caudalanteriorcingulate
850,2003,ctx-caudalmiddlefrontal
851,2004,ctx-corpuscallosum
852,2005,ctx-cuneus
853,2006,ctx-entorhinal
854,2007,ctx-fusiform


In [9]:
# The cortical names have NO hemisphere tag: 'ctx-superiorfrontal', not 'ctx-rh-superiorfrontal'.
# Notebooks 02 and 04 show why this matters.
print(df[df.ID.isin([2000, 2028])].to_string(index=False))

  ID                NAME
2000         ctx-unknown
2028 ctx-superiorfrontal


## The other bundled LUTs

`LUT.load(path)` (same as `load_lut(path)`) reads any FreeSurfer-format table:

In [10]:
rows = []
for f in sorted(LUTDIR.glob("*.txt")):
    t = io.load_lut(f)
    rows.append((f.name, len(t), int(t["ID"].min()), int(t["ID"].max())))
pd.DataFrame(rows, columns=["file", "n_labels", "min_id", "max_id"])

,file,n_labels,min_id,max_id
0,ASegLUT.txt,105,0,2035
1,FreeSurferColorLUT.txt,1804,0,15137
2,LauraLUT.txt,28,0,114
3,NextBrainLUT.txt,883,0,2035
4,SuperSynthCerebrumLUT.txt,36,0,870
5,SuperSynthExVivoLUT.txt,43,0,870
6,SuperSynthWholeLUT.txt,56,0,930


In [11]:
laura = io.load_lut(LUTDIR / "LauraLUT.txt")
pd.DataFrame({"ID": laura.labels, "NAME": laura.names})

,ID,NAME
0,0,Unknown
1,1,Left-Cerebral-White-Matter
2,2,Left-Cerebral-Grey-Matter
3,3,Left-Cerebral-Cortex
4,4,Left-Cerebellum-White-Matter
5,5,Left-Cerebellum-Cortex
6,6,Left-Thalamus
7,7,Left-Caudate-Accumbens-area
8,8,Left-Putamen
9,9,Left-Pallidum


Notice the "Laura" scheme: left labels are 1–14 and right labels are 101–114
(offset by 100). Keep this in mind for the bug in notebook 06.

## YAML mapping files (`load_yaml`)

Each mapping YAML has the form `TargetName: [list of NextBrain IDs]`. Target
names are written **without** `Left-`/`Right-`. The converter adds the prefix
for lateralised structures (notebook 06).

In [12]:
aseg_map = io.load_yaml(LUTDIR / "NextBrainToASeg.yaml")
for k, v in aseg_map.items():
    print(f"{k:25s} {len(v or []):4d} NextBrain labels   e.g. {(v or [])[:6]}")

Cerebral-White-Matter       13 NextBrain labels   e.g. [7, 68, 130, 100, 161, 199]
Cerebral-Cortex              6 NextBrain labels   e.g. [99, 113, 125, 157, 238, 240]
Cerebellum-White-Matter      5 NextBrain labels   e.g. [846, 721, 751, 752, 715]
Cerebellum-Cortex            2 NextBrain labels   e.g. [595, 597]
Thalamus                    69 NextBrain labels   e.g. [218, 219, 252, 220, 378, 379]
Caudate                      4 NextBrain labels   e.g. [48, 118, 393, 184]
Putamen                      2 NextBrain labels   e.g. [79, 349]
Pallidum                     2 NextBrain labels   e.g. [119, 206]
Hippocampus                 54 NextBrain labels   e.g. [364, 365, 367, 404, 405, 407]
Amygdala                    13 NextBrain labels   e.g. [214, 215, 277, 278, 238, 279]
Accumbens-area               1 NextBrain labels   e.g. [101]
VentralDC                   43 NextBrain labels   e.g. [310, 315, 316, 321, 352, 385]
Claustrum                    2 NextBrain labels   e.g. [102, 174]
Brain-St

### Coverage check: which *used* NextBrain labels does each YAML **not** mention?

Because the converters start from the identity mapping, an unlisted label
**keeps its NextBrain ID**. In the output, that ID would then be read as some
unrelated ASeg or SuperSynth label, or as an unknown one. We check against
`NextBrainLabels.yaml`, i.e. the labels that really occur.

In [13]:
used = io.load_yaml(LUTDIR / "NextBrainLabels.yaml")
print(len(used), "entries; note one value is a list:", {k: v for k, v in used.items() if isinstance(v, list)})
used_ids = {i for v in used.values() for i in (v if isinstance(v, list) else [v])}
sub = {i for i in used_ids if 0 < i < 1000}                 # cortex is handled separately
print(len(sub), "subcortical labels used by NextBrain\n")
for f in ["NextBrainToASeg.yaml", "NextBrainToSuperSynth.yaml", "NextBrainToLaura.yaml"]:
    m = io.load_yaml(LUTDIR / f)
    listed = {x for v in m.values() for x in (v or [])}
    missing = sorted(sub - listed)
    print(f"{f:28s} lists {len(listed):3d}; used-but-unlisted: {len(missing):2d}",
          [(i, lut.label2name(i)[:30]) for i in missing[:3]])

334 entries; note one value is a list: {'white_matter_of_hindbrain': [846, 611]}
300 subcortical labels used by NextBrain

NextBrainToASeg.yaml         lists 274; used-but-unlisted: 27 [(386, 'root_of_oculomotor_nerve'), (401, 'parabrachial_pigmented_nucleus'), (403, 'cerebellothalamic_tract')]
NextBrainToSuperSynth.yaml   lists 271; used-but-unlisted: 29 [(50, 'anterior_horn_of_lateral_ventr'), (114, 'optic_nerve'), (386, 'root_of_oculomotor_nerve')]
NextBrainToLaura.yaml        lists 228; used-but-unlisted: 72 [(50, 'anterior_horn_of_lateral_ventr'), (120, 'ventral_pallidus'), (386, 'root_of_oculomotor_nerve')]


## The Allen ontology (`load_ontology`)

The Allen ontology is a **tree** of nested dictionaries. Each node has `id`,
`acronym`, `name`, `color_hex_triplet`, `parent_structure_id`, `children`, and
a few other fields. Notebook 02 explores it in detail.

In [14]:
ont = io.load_ontology()          # default = AllenBrainOntologyDev.json
print({k: v for k, v in ont.items() if k != "children"})
print("children of the root:", [c["name"] for c in ont["children"]])

{'id': 10153, 'atlas_id': None, 'ontology_id': 11, 'acronym': 'NP', 'name': 'neural plate', 'color_hex_triplet': 'D7D8D8', 'graph_order': 0, 'st_level': None, 'hemisphere_id': 3, 'parent_structure_id': None}
children of the root: ['neural tube']


Next: **02 · The Allen ontology and `lut.py`**.